# Auditing components.yaml

This notebook runs the AuditSystem against the main components.yaml file to evaluate solution quality.

In [ ]:
from pathlib import Path

import ibis

from iacs.audit_system import (
    AuditRunner,
    RequirementCoverageAudit,
    TraceabilityAudit,
    TodoAudit,
)
from iacs.transforms.manifest_to_registry import (
    raw_entity_first_data,
    flattened_entity_first_data,
    component_first_data,
    complete_schema,
    data_models,
    components_database,
    validated_components,
    registry,
)

ibis.options.interactive = True  # auto-display results (like pandas)

## Load components.yaml

In [ ]:
COMPONENTS_DIR = str(Path("../components"))

raw = raw_entity_first_data(COMPONENTS_DIR)
flat = flattened_entity_first_data(raw)
comp_first = component_first_data(flat)
schema = complete_schema(comp_first["schema"], comp_first["parent"])
models = data_models(schema)
conn, comps = components_database(comp_first, models)
v_comps = validated_components(comps, models)
reg = registry(conn, v_comps)

print(f"Component types: {reg.component_types}")
print(f"Number of component types: {len(reg.component_types)}")

## Run All Audits

In [ ]:
runner = AuditRunner(audits=[
    RequirementCoverageAudit(),
    TraceabilityAudit(),
    TodoAudit(),
])

results = runner.run(reg)

print(f"All audits passed: {runner.all_passed}")
print()
for name, result in results.items():
    status = "PASS" if result.passed else "FAIL"
    print(f"{name}: {status}")

## Requirement Coverage Audit

Checks that all requirements have implementing solutions.

In [ ]:
reg.view(["requirement", "description"])

In [ ]:
results["requirement_coverage"].view()

## Traceability Audit

Checks that all entities trace back to requirements.

In [ ]:
trace_result = results["traceability"]
print(f"Passed: {trace_result.passed}")
print(f"Orphan entities: {len(trace_result.results)}")
print()
if not trace_result.results.empty:
    print("Entities not tracing to requirements:")
    trace_result.view()

## Todo Audit

Reports outstanding todos.

In [ ]:
todo_result = results["todo"]
print(f"Passed: {todo_result.passed}")
print(f"Entities with todos: {len(todo_result.results)}")
print()
if todo_result.messages:
    print("Outstanding todos:")
    for msg in todo_result.messages:
        print(f"  - {msg}")